In [2]:
# ══════════════════════════════════════════════════════
# CELL 1 — Install packages (run first, takes ~3 mins)
# ══════════════════════════════════════════════════════

%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets

In [3]:
# ══════════════════════════════════════════════════════
# CELL 2 — Verify GPU
# ══════════════════════════════════════════════════════

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

GPU: Tesla T4
VRAM: 15.6 GB
CUDA: 12.8


In [4]:
# ══════════════════════════════════════════════════════
# CELL 3 — Load Phi-3-mini in 4-bit (QLoRA)
# ══════════════════════════════════════════════════════

from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 1024
DTYPE          = None   # auto-detect
LOAD_IN_4BIT   = True   # QLoRA — uses ~4GB VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Phi-3-mini-4k-instruct",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
)
print("✅ Model loaded")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.5: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

✅ Model loaded


In [5]:
# ══════════════════════════════════════════════════════
# CELL 4 — Apply LoRA adapters
# ══════════════════════════════════════════════════════

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,          # LoRA rank — higher = more capacity
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

# Print trainable parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")


Unsloth 2026.5.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Total params:     2,039,024,640
Trainable params: 29,884,416 (1.47%)


In [6]:
# ══════════════════════════════════════════════════════
# CELL 5 — Upload and prepare dataset
# ══════════════════════════════════════════════════════

# Upload json_extraction_dataset.jsonl from your local machine
from google.colab import files
uploaded = files.upload()  # select json_extraction_dataset.jsonl

import json

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

raw_data = load_jsonl("json_extraction_dataset.jsonl")
print(f"✅ Loaded {len(raw_data)} training examples")

# Format for Phi-3 chat template
PROMPT_TEMPLATE = """<|user|>
Extract structured JSON from the following text. Output ONLY valid JSON, nothing else.

Text: {input}
<|end|>
<|assistant|>
{output}<|end|>
"""

def format_example(example):
    return {
        "text": PROMPT_TEMPLATE.format(
            input=example["input"],
            output=example["output"]
        )
    }

from datasets import Dataset
dataset = Dataset.from_list([format_example(d) for d in raw_data])
print(f"Dataset ready: {len(dataset)} examples")
print("\nSample:")
print(dataset[0]["text"][:300])


Saving json_extraction_dataset.jsonl to json_extraction_dataset.jsonl
✅ Loaded 25 training examples
Dataset ready: 25 examples

Sample:
<|user|>
Extract structured JSON from the following text. Output ONLY valid JSON, nothing else.
 
Text: John Smith ordered 3 laptops at $999 each on March 15th. Ship to 123 Main St, Boston. Delivery in 5 business days.
<|end|>
<|assistant|>
{"customer_name": "John Smith", "item": "laptop", "quantity


In [7]:
# ══════════════════════════════════════════════════════
# CELL 6 — BEFORE fine-tuning: evaluate base model
# ══════════════════════════════════════════════════════

import sys, re
sys.path.append('/content')

# Upload evaluate.py too
print("Upload evaluate.py now...")
uploaded2 = files.upload()

from evaluate import TEST_CASES, PROMPT_TEMPLATE as EVAL_PROMPT, compare_and_save

# Switch model to inference mode
FastLanguageModel.for_inference(model)

INFERENCE_PROMPT = """<|user|>
Extract structured JSON from the following text. Output ONLY valid JSON, nothing else.

Text: {input}
<|end|>
<|assistant|>
"""

def generate(text, max_new_tokens=300):
    prompt = INFERENCE_PROMPT.format(input=text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only new tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("Running BEFORE evaluation (5 test cases)...")
before_outputs = []
for i, test in enumerate(TEST_CASES):
    out = generate(test["input"])
    before_outputs.append(out)
    print(f"  [{i+1}] {out[:80]}...")

print("\n✅ Before outputs collected")

Upload evaluate.py now...


Saving evaluate.py to evaluate.py
Running BEFORE evaluation (5 test cases)...


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

  [1] ```json

{

  "customer": "Alex Turner",

  "order_number": "55821",

  "items":...


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [2] ```json

{

  "Event": "Annual Tech Summit",

  "Date": "September 20-22",

  "V...


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [3] ```json

{

  "SoftwarePackage": {

    "Name": "numpy",

    "Version": "1onian...


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [4] ```json

{

  "hotel_booking": {

    "name": "Marriott Downtown",

    "room": ...
  [5] ```json

{

  "pull_request": {

    "id": 342,

    "author": "dev-john",

    ...

✅ Before outputs collected


In [8]:
# ══════════════════════════════════════════════════════
# CELL 7 — Train with SFTTrainer
# ══════════════════════════════════════════════════════

from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Switch back to training mode
FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = dataset,
    dataset_text_field = "text",
    max_seq_length   = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing          = True,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps        = 5,
        num_train_epochs    = 3,       # 3 passes over 25 examples
        learning_rate       = 2e-4,
        fp16                = not is_bfloat16_supported(),
        bf16                = is_bfloat16_supported(),
        logging_steps       = 5,
        optim               = "adamw_8bit",
        weight_decay        = 0.01,
        lr_scheduler_type   = "linear",
        seed                = 42,
        output_dir          = "./lora_output",
        report_to           = "none",
    ),
)

print("🚀 Starting training...")
import time
t0 = time.time()
trainer_stats = trainer.train()
elapsed = round(time.time() - t0)
print(f"\n✅ Training complete in {elapsed}s")
print(f"   Final loss: {trainer_stats.training_loss:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/25 [00:00<?, ? examples/s]

🚀 Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 25 | Num Epochs = 3 | Total steps = 12
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,1.166654
10,0.862782


Unsloth: Restored added_tokens_decoder metadata in ./lora_output/checkpoint-12/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./lora_output/checkpoint-12.



✅ Training complete in 35s
   Final loss: 0.9870


In [9]:
# ══════════════════════════════════════════════════════
# CELL 8 — AFTER fine-tuning: evaluate + compare
# ══════════════════════════════════════════════════════

# Switch to inference mode
FastLanguageModel.for_inference(model)

print("Running AFTER evaluation (5 test cases)...")
after_outputs = []
for i, test in enumerate(TEST_CASES):
    out = generate(test["input"])
    after_outputs.append(out)
    print(f"  [{i+1}] {out[:80]}...")

# Compare and save metrics
results = compare_and_save(before_outputs, after_outputs)

Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running AFTER evaluation (5 test cases)...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [1] {"customer_name": "Alex Turner", "order_number": "55821", "product": "noise-canc...


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [2] {"event_name": "Annual Tech Summit", "date": "2021-09-20/2021-09-22", "venue": "...


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [3] {"software_package": "numpy", "version": "1limp_1.26.4", "language": "Python", "...


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [4] {"hotel_name": "Marriott Downtown", "room_type": "Deluxe King", "check_in": "Oct...
  [5] {"pull_request": {"id": 342, "user": "dev-john", "repo": "backend-api", "title":...

  BEFORE FINE-TUNING
  [1] valid=True | accuracy=0.29 | compliance=0.5
  [2] valid=True | accuracy=0.00 | compliance=0.5
  [3] valid=False | accuracy=0.00 | compliance=0.5
  [4] valid=True | accuracy=0.00 | compliance=0.5
  [5] valid=True | accuracy=0.00 | compliance=0.5

  ────────────────────────────────────────
  Valid JSON:    80.0%
  Field Accuracy:5.7%
  Format Score:  50.0%
  Overall:       0.452 / 1.000

  AFTER FINE-TUNING (LoRA)
  [1] valid=True | accuracy=0.71 | compliance=1.0
  [2] valid=True | accuracy=0.44 | compliance=1.0
  [3] valid=True | accuracy=0.42 | compliance=1.0
  [4] valid=True | accuracy=0.38 | compliance=1.0
  [5] valid=True | accuracy=0.00 | compliance=1.0

  ────────────────────────────────────────
  Valid JSON:    100.0%
  Field Accuracy:38.9%
  Format Score:  100.0%
  Overall: 

In [10]:
# ══════════════════════════════════════════════════════
# CELL 9 — Save LoRA weights + metrics to Drive
# ══════════════════════════════════════════════════════

import os, shutil
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/phi3_json_lora"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save LoRA adapter weights
model.save_pretrained(f"{SAVE_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{SAVE_DIR}/lora_adapter")
print(f"✅ LoRA adapter saved to Drive")

# Save metrics
import json
with open(f"{SAVE_DIR}/metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"✅ Metrics saved to Drive")

# Download metrics.json locally
files.download("results/metrics.json")
print("\n📥 metrics.json downloaded — commit to GitHub")

Mounted at /content/drive


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/phi3_json_lora/lora_adapter/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/phi3_json_lora/lora_adapter.


✅ LoRA adapter saved to Drive
✅ Metrics saved to Drive


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📥 metrics.json downloaded — commit to GitHub


In [11]:
# ══════════════════════════════════════════════════════
# CELL 10 — Print final summary for README
# ══════════════════════════════════════════════════════

b = results["before"]
a = results["after"]
imp = results["improvement"]

print("\n" + "="*55)
print("  COPY THIS INTO YOUR README:")
print("="*55)
print(f"""
| Metric | Base Model | Fine-Tuned (LoRA) | Improvement |
|---|---|---|---|
| Valid JSON Rate | {b['valid_json_rate']:.0%} | {a['valid_json_rate']:.0%} | +{imp['valid_json_improvement']:.0%} |
| Field Accuracy | {b['field_accuracy']:.0%} | {a['field_accuracy']:.0%} | +{imp['field_accuracy_improvement']:.0%} |
| Format Compliance | {b['format_compliance']:.0%} | {a['format_compliance']:.0%} | +{imp['format_improvement']:.0%} |
| **Overall Score** | **{b['overall_score']:.3f}** | **{a['overall_score']:.3f}** | **+{imp['overall_improvement']:.3f}** |
""")



  COPY THIS INTO YOUR README:

| Metric | Base Model | Fine-Tuned (LoRA) | Improvement |
|---|---|---|---|
| Valid JSON Rate | 80% | 100% | +20% |
| Field Accuracy | 6% | 39% | +33% |
| Format Compliance | 50% | 100% | +50% |
| **Overall Score** | **0.452** | **0.796** | **+0.344** |

